# DubsiBhai: test the trained Bengali classifier

Load the saved model, try your own complaints, compare example predictions, and inspect the training report. **The API server is not required** for the local model cells.

## Setup
From `apps/ml-service` in PowerShell:
```powershell
uv sync --locked --group notebooks
# Only if no trained model exists yet:
./train.ps1
```
Open this notebook in VS Code, choose **Select Kernel → Python Environments**, and select `apps/ml-service/.venv/Scripts/python.exe`. Run the cells from top to bottom.

This notebook does not retrain or overwrite your model. Load only artifacts generated by your own training command.

In [1]:
import json
import sys
from html import escape
from pathlib import Path

from IPython.display import HTML, display

# Works when opened from the repository root, the service, or its notebooks folder.
service_root = None
for directory in (Path.cwd(), *Path.cwd().parents):
    candidates = (directory, directory / "apps" / "ml-service")
    for candidate in candidates:
        if (candidate / "app" / "models" / "classifier.py").is_file():
            service_root = candidate.resolve()
            break
    if service_root is not None:
        break
if service_root is None:
    raise RuntimeError("Open the notebook from inside the DubsiBhai repository.")
if str(service_root) not in sys.path:
    sys.path.insert(0, str(service_root))

expected_environment = service_root / ".venv"
print(f"Notebook Python: {sys.executable}")
if Path(sys.prefix).resolve() != expected_environment.resolve():
    raise RuntimeError(
        "Wrong notebook kernel. In VS Code choose Select Kernel > Python Environments > "
        f"{expected_environment / 'Scripts' / 'python.exe'}, then Restart Kernel and Run All. "
        "Do not retrain the model in this unrelated environment."
    )

from app.core.config import get_settings
from app.models.classifier import ComplaintClassifier

get_settings.cache_clear()
model_path = get_settings().model_path
if not model_path.is_absolute():
    model_path = service_root / model_path
if not model_path.is_file():
    raise FileNotFoundError("No trained model found. Run ./train.ps1 inside apps/ml-service first.")
model = ComplaintClassifier.load(model_path)
print(f"Model: {model_path}")
print(f"Version: {model.metadata['model_version']}")
print(f"Training source: {model.metadata['data_source']}")

Notebook Python: d:\CSE-4113-Project\apps\ml-service\.venv\Scripts\python.exe
Model: D:\CSE-4113-Project\apps\ml-service\weights\classifier.joblib
Version: 20260925T202213758971Z-b958a49536db
Training source: synthetic


## 1. Try your own complaint
Edit `complaint` below and rerun this cell and the probability chart. The response uses the same predictor as `POST /classify`.

In [2]:
complaint = "ম্যানহোলের ঢাকনা নেই, বাচ্চারা পড়ে যেতে পারে।"
result = model.predict(complaint)
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "predicted_category": "open_manhole",
  "confidence": 0.8068173737944686,
  "probabilities": {
    "blocked_drain": 0.03767390549951383,
    "open_manhole": 0.8068173737944686,
    "other": 0.03813616023576267,
    "road_flooding": 0.03992531041117315,
    "sewage_overflow": 0.04285557550329673,
    "waterlogging_recurring": 0.03459167455578496
  },
  "model_version": "20260925T202213758971Z-b958a49536db",
  "data_source": "synthetic"
}


In [3]:
# A probability chart without additional plotting dependencies.
rows = []
for category, probability in sorted(result["probabilities"].items(), key=lambda item: item[1], reverse=True):
    rows.append(
        f"<tr><td>{escape(category)}</td>"
        f"<td><progress max='1' value='{probability}'></progress></td>"
        f"<td>{probability:.1%}</td></tr>"
    )
display(HTML("<table><caption>Category probabilities</caption><tbody>" + "".join(rows) + "</tbody></table>"))
print("Confidence is an uncalibrated model probability, not a guarantee of correctness.")

open_manhole,,80.7%
sewage_overflow,,4.3%
road_flooding,,4.0%
other,,3.8%
blocked_drain,,3.8%
waterlogging_recurring,,3.5%


Confidence is an uncalibrated model probability, not a guarantee of correctness.


## 2. Compare several examples
These expected labels are manual examples for exploration, **not an independent test set or an accuracy benchmark**. Replace them with your own text to explore errors and ambiguous complaints.

In [4]:
examples = [
    ("ড্রেনে ময়লা জমে পানি চলাচল বন্ধ হয়ে গেছে।", "blocked_drain"),
    ("ম্যানহোলের ঢাকনা নেই, বাচ্চারা পড়ে যেতে পারে।", "open_manhole"),
    ("বৃষ্টির পর রাস্তায় হাঁটু পানি, যানবাহন চলতে পারছে না।", "road_flooding"),
    ("প্রতি বছর একই রাস্তায় পানি জমে, স্থায়ী সমাধান দরকার।", "waterlogging_recurring"),
    ("স্যুয়ারেজ লাইন ফেটে রাস্তায় নোংরা পানি উপচে পড়ছে।", "sewage_overflow"),
    ("নতুন রাস্তা নির্মাণের সময় ড্রেন ব্যবস্থা ঠিকমতো করা হয়নি।", "other"),
]
rows = []
for text, expected in examples:
    prediction = model.predict(text)
    values = [text, expected, prediction["predicted_category"],
              f"{prediction['confidence']:.1%}",
              "Match" if prediction["predicted_category"] == expected else "Review"]
    rows.append("<tr>" + "".join(f"<td>{escape(value)}</td>" for value in values) + "</tr>")
display(HTML(
    "<table><thead><tr><th>Complaint</th><th>Expected</th><th>Predicted</th>"
    "<th>Confidence</th><th>Comparison</th></tr></thead><tbody>"
    + "".join(rows) + "</tbody></table>"
))

Complaint,Expected,Predicted,Confidence,Comparison
ড্রেনে ময়লা জমে পানি চলাচল বন্ধ হয়ে গেছে।,blocked_drain,blocked_drain,29.0%,Match
"ম্যানহোলের ঢাকনা নেই, বাচ্চারা পড়ে যেতে পারে।",open_manhole,open_manhole,80.7%,Match
"বৃষ্টির পর রাস্তায় হাঁটু পানি, যানবাহন চলতে পারছে না।",road_flooding,road_flooding,52.1%,Match
"প্রতি বছর একই রাস্তায় পানি জমে, স্থায়ী সমাধান দরকার।",waterlogging_recurring,waterlogging_recurring,74.5%,Match
স্যুয়ারেজ লাইন ফেটে রাস্তায় নোংরা পানি উপচে পড়ছে।,sewage_overflow,sewage_overflow,87.6%,Match
নতুন রাস্তা নির্মাণের সময় ড্রেন ব্যবস্থা ঠিকমতো করা হয়নি।,other,other,73.6%,Match


## 3. Inspect the stored evaluation
The report is embedded in the loaded artifact, so it matches this model version. Evaluation fitted the vectorizer and classifier on the training split only; the serving model was subsequently refitted on all unique rows.

Exact duplicates were removed before splitting. Synthetic template variants may still appear in both splits. High scores on synthetic text do not establish performance on real complaints.

In [5]:
report = model.metadata
evaluation = report["evaluation"]
summary = {key: report[key] for key in (
    "trained_at", "data_source", "dataset", "dataset_sha256", "input_rows",
    "unique_rows", "duplicates_removed", "serving_model_training_rows"
)}
summary.update({key: evaluation[key] for key in ("train_rows", "test_rows", "accuracy", "macro_f1")})
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nLimitations:", evaluation["limitations"])

{
  "trained_at": "2026-09-25T20:22:13.758971+00:00",
  "data_source": "synthetic",
  "dataset": "drainage_complaints_bn.csv",
  "dataset_sha256": "b958a49536db6320cd2104a3ef5eeb54ecff847e4eb85bd0f38307d890269772",
  "input_rows": 360,
  "unique_rows": 295,
  "duplicates_removed": 65,
  "serving_model_training_rows": 295,
  "train_rows": 236,
  "test_rows": 59,
  "accuracy": 0.9830508474576272,
  "macro_f1": 0.9832915622389308
}

Limitations: Random holdout scores do not establish real-world accuracy. Synthetic template variants may occur in both splits. Evaluate on independent real complaints before drawing deployment conclusions. Confidence values are not calibrated.


In [6]:
labels = evaluation["confusion_matrix_labels"]
header = "<tr><th>Actual / Predicted</th>" + "".join(f"<th>{escape(label)}</th>" for label in labels) + "</tr>"
rows = []
for label, counts in zip(labels, evaluation["confusion_matrix"], strict=True):
    rows.append(f"<tr><th>{escape(label)}</th>" + "".join(f"<td>{count}</td>" for count in counts) + "</tr>")
display(HTML("<table><caption>Stored holdout confusion matrix</caption><thead>" + header + "</thead><tbody>" + "".join(rows) + "</tbody></table>"))

Actual / Predicted,blocked_drain,open_manhole,road_flooding,waterlogging_recurring,sewage_overflow,other
blocked_drain,9,0,0,0,0,0
open_manhole,0,9,0,0,0,0
road_flooding,0,0,9,1,0,0
waterlogging_recurring,0,0,0,10,0,0
sewage_overflow,0,0,0,0,11,0
other,0,0,0,0,0,10


## 4. Optional: test the running API
Start `./run.ps1` in a separate terminal inside `apps/ml-service`, then set `TEST_API = True` below. This sends an HTTP request to the live service. A different model version means the server or notebook may need restarting after retraining.

In [7]:
import httpx

TEST_API = False
API_URL = "http://127.0.0.1:8001"
if TEST_API:
    try:
        response = httpx.post(f"{API_URL}/classify", json={"text": complaint}, timeout=15)
        response.raise_for_status()
        api_result = response.json()
        print(json.dumps(api_result, ensure_ascii=False, indent=2))
        print("Same model version:", api_result["model_version"] == model.metadata["model_version"])
    except httpx.ConnectError:
        print("Start ./run.ps1 in apps/ml-service, then rerun this cell.")
    except httpx.HTTPStatusError as error:
        print(f"API returned {error.response.status_code}: {error.response.text}")
    except httpx.TimeoutException:
        print("The API request timed out. Check the service logs and try again.")
else:
    print("Optional live API test skipped. Set TEST_API = True to enable it.")

Optional live API test skipped. Set TEST_API = True to enable it.


## 5. Retrain later with real data
Keep the UTF-8 CSV columns `id,text_bn,category` and the same six category names. From `apps/ml-service`:
```powershell
./train.ps1 -Dataset ../../docs/dataset/real_complaints_bn.csv -DataSource real
```
Or replace the default CSV and run `./train.ps1 -DataSource real`.

Restart the API and **restart this notebook's kernel**, then run all cells to load the new artifact. Keep a separate real-world evaluation set before making performance claims. See [the ML service README](../README.md) for the full training workflow.